# Learn AutoGluon in 60 Minutes

## Learning goals
By the end of this guide, you should be able to:
1. Explain what AutoGluon automates and what you still control.
2. Create a `TabularPredictor` and fit it on labeled tabular data.
3. Predict unseen prices, compare models, and evaluate errors.
4. Calculate permutation feature importance and interpret its limitations.
5. Save/load a trained predictor.


## 1. What is AutoGluon?
AutoGluon is an AutoML library that can fit and compare multiple algorithms, automate their preprocessing and tuning, and combine models into an ensemble. Unlike a hand-written scikit-learn pipeline, AutoGluon can infer how to handle many categorical and text features, reducing amount of preprocessing necessary. 

## 2. API quick reference
| API | Purpose | Important parameter(s) |
|---|---|---|
| `TabularPredictor(...)` | Define prediction problem | `label`, `problem_type`, `eval_metric`, `path` |
| `.fit(train_data, ...)` | Train candidate models and ensemble | `presets`, `time_limit` |
| `.predict(features)` | Predict target for new rows | Feature columns |
| `.evaluate(test_data)` | Evaluate labeled test observations | Data must include `label` |
| `.leaderboard(test_data)` | Compare trained model performance | Optional test data |
| `.feature_importance(test_data)` | Permutation-based importance | Labeled evaluation dataset |
| `TabularPredictor.load(path)` | Reopen saved model | Predictor directory |

**A note on metrics:** In AutoGluon, MAE-based leaderboard values may be represented in a higher-is-better form (negative MAE). A *less negative* score is better; use standard MAE calculated with scikit-learn when reporting dollar errors.

## 3. Minimal working demonstration
This example is runnable without downloading the Airbnb dataset, as it is designed for learning the API quickly.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from autogluon.tabular import TabularPredictor

rng = np.random.default_rng(42)
n = 240
rooms = rng.choice(["Entire home/apt", "Private room"], size=n)
boroughs = rng.choice(["Manhattan", "Brooklyn", "Queens"], size=n)
price = (55 + 100 * (rooms == "Entire home/apt") + 55 * (boroughs == "Manhattan")
         + rng.normal(0, 18, size=n))
demo = pd.DataFrame({"room_type": rooms, "borough": boroughs, "minimum_nights": rng.integers(1, 15, size=n), "price": price})
train_demo = demo.sample(frac=0.8, random_state=42)
test_demo = demo.drop(train_demo.index)
print(train_demo.shape, test_demo.shape)

(192, 4) (48, 4)


### `TabularPredictor` and `.fit()`
`label="price"` selects the target. `problem_type="regression"` predicts a numeric quantity. `eval_metric="mean_absolute_error"` prioritizes dollar-scale errors. `path` specifies where the predictor saves its artifacts. `time_limit` keeps this teaching example relatively short. It can also lead to different trained model sets.

In [2]:
demo_path = "AutogluonModels/tutorial_demo"
demo_predictor = TabularPredictor(
    label="price",
    problem_type="regression",
    eval_metric="mean_absolute_error",
    path=demo_path,
).fit(train_data=train_demo, time_limit=90)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.11.17
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.13.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       0.80 GB / 7.92 GB (10.1%)
Disk Space Avail:   5.02 GB / 236.34 GB (2.1%)
	We recommend a minimum available disk space of 10 GB, and large datasets may require more.
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best' on datasets <100000 samples by using Tabular Foundation Model

### `.predict()`, `.evaluate()`, and `.leaderboard()`
`.predict()` uses **only the features** to produce target predictions on the test data. Does not require explicitly dropping the target column from the test data; AutoGluon will ignore it if present. `.evaluate()` computes how well the model was able to make predictions across several metrics. `.leaderboard()` provides a comparison table of the performances of the different models that AutoGluon explored.

`.evaluate()` requires labeled test data. Do not confuse the leaderboard's internal validation performance with unseen test performance.

In [3]:
demo_preds = demo_predictor.predict(test_demo)
print("First predictions:", demo_preds.head())
print("Test evaluation:", demo_predictor.evaluate(test_demo))
display(demo_predictor.leaderboard(test_demo, silent=True).head())

First predictions: 1      43.542503
14     56.118423
20     48.878105
21    187.987671
37     48.878105
Name: price, dtype: float32
Test evaluation: {'mean_absolute_error': -16.664398307335787, 'root_mean_squared_error': np.float64(-19.97372991355191), 'mean_squared_error': -398.9498866595184, 'r2': 0.8941657959296199, 'pearsonr': 0.9479604253789606, 'median_absolute_error': -15.345362686421716}


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBM,-14.117995,-17.421103,mean_absolute_error,0.060618,0.015494,0.918296,0.060618,0.015494,0.918296,1,True,2
1,LightGBMXT,-14.142607,-17.376292,mean_absolute_error,0.052674,0.079208,0.392768,0.052674,0.079208,0.392768,1,True,1
2,NeuralNetTorch,-15.339573,-16.656226,mean_absolute_error,0.037629,0.015688,12.382374,0.037629,0.015688,12.382374,1,True,8
3,CatBoost,-16.144668,-18.424794,mean_absolute_error,0.034866,0.015345,24.654255,0.034866,0.015345,24.654255,1,True,4
4,NeuralNetFastAI,-16.471268,-16.694062,mean_absolute_error,0.110285,0.033210,12.606422,0.110285,0.033210,12.606422,1,True,6


### `.feature_importance()`
Feature importance measures how much predictive performance worsens when a feature’s values are shuffled. It measures predictive dependence, not causation. Larger positive values indicate greater importance, while values near zero mean shuffling had little effect. Negative values mean performance improved after shuffling, which may suggest noise or a potentially harmful feature. Correlated features like square footage and number of bedrooms, can substitute for each other, making their individual importance seem lower.

In [4]:
demo_importance = demo_predictor.feature_importance(test_demo, num_shuffle_sets=3, silent=True)
display(demo_importance)

,importance,stddev,p_value,n,p99_high,p99_low
room_type,40.113446,5.944594,0.003621,3,74.176628,6.050263
borough,17.589539,4.218437,0.009319,3,41.761650,-6.582573
minimum_nights,-1.026453,0.476437,0.967549,3,1.703583,-3.756488


### Model persistence
The predictor saves its artifacts to previously specified path: `demo_path`. You can reload the predictor as long as the files still exist. For a new experiment, it is suggested to create a new path, as re-running `.fit()` at an existing path can cause conflicts with old artifacts.

In [5]:
reloaded = TabularPredictor.load(demo_path)
assert len(reloaded.predict(test_demo.drop(columns="price"))) == len(test_demo)
print("Reload successful")

Reload successful


## Checkpoint questions
1. What does `.predict()` indicate? What does a greater value signify?
2. What is the difference between `.evaluate()` and `.leaderboard()`?
3. What changes when you decrease `time_limit`?
4. Train a new predictor using `eval_metric="root_mean_squared_error"`. Keep the same training data and time limit, and save it to a new path `AutogluonModels/rmseCheckpoint`. Which three models have the best validation scores? Compare their ranking with the original MAE-based leaderboard and discuss any differences. **Feel free to use the cells below**

**Answer Here:**


**Next:** Open `autogluon.example.ipynb` to apply these methods to real NYC Airbnb data.